# game
## payoff matrix
T,R,P,S = 3,2,1,0

## Preparation

In [1]:
import random
import math

In [2]:
C, D = 0, 1

PAYOFF = [
    [(2, 2), (0, 3)],
    [(3, 0), (1, 1)]
]

class MatchState:
    def __init__(self, N=5, T=100, e = 0.01):
        self.N = N
        self.T = T
        self.e = e
        self.reset_match()
    
    def set_strategies(self, s1, s2):
        self.strategy1_name = str(s1)
        self.strategy2_name = str(s2)

    def reset_game(self):
        if self.history:
            self.all_history.append({
                "strategy1": self.strategy1_name,
                "strategy2": self.strategy2_name,
                "history": self.history.copy()
            })
        self.history = []
        self.t = 0
        self.n += 1

    def reset_match(self):
        self.history = []
        self.all_history = []
        self.t = 0
        self.n = 0


    def step(self, a, b):
        # trembling hand noise (independent per player)
        if random.random() < self.e:
            a = 1 - a  # flip action of player 1

        if random.random() < self.e:
            b = 1 - b  # flip action of player 2

        r1, r2 = PAYOFF[a][b]

        self.history.append((a, b, r1, r2))
        self.t += 1

        return r1, r2

    def is_game_over(self):
        return self.t >= self.T

    def is_match_over(self):
        return self.n >= self.N
    
    def match_play(self, strategy1, strategy2):
        self.reset_match()
        self.set_strategies(strategy1, strategy2)

        while not self.is_match_over():
            self.reset_game()
            strategy1.reset()
            strategy2.reset()

            while not self.is_game_over():
                a = strategy1.act(self)
                b = strategy2.act(self)
                self.step(a, b)

        # save last game
        if self.history:
            self.all_history.append({
                "strategy1": self.strategy1_name,
                "strategy2": self.strategy2_name,
                "history": self.history.copy()
            })

        return self.all_history


## Population initiation



In [3]:
class Strategy:
    """
    Base class for IPD strategies.

    Contract:
    - act(state) must return 0 (C) or 1 (D)
    - reset() must clear internal state between matches
    """
    def __str__(self):
        return self.__class__.__name__

    COOP = 0
    DEFECT = 1

    def reset(self):
        """Reset internal state before a new match."""
        pass

    def act(self, state):
        """
        Decide next action.

        Parameters:
            state:
                state.history -> list of (my_action, opp_action)
                state.turn   -> int

        Returns:
            int: 0 (COOP) or 1 (DEFECT)
        """
        raise NotImplementedError

### Finite State Machine

In [4]:
class TitForTat(Strategy):
    def act(self, state):
        if not state.history:
            return self.COOP
        return state.history[-1][1]  # opponent's last move
    
class TitForTwoTats(Strategy):
    def act(self, state):
        if len(state.history) < 2:
            return self.COOP

        last_two = state.history[-2:]
        if last_two[0][1] == self.DEFECT and last_two[1][1] == self.DEFECT:
            return self.DEFECT

        return self.COOP
    
class GrimTrigger(Strategy):
    def reset(self):
        self.triggered = False

    def act(self, state):
        if self.triggered:
            return self.DEFECT

        # check if opponent ever defected
        if any(o == self.DEFECT for _, o, _, _ in state.history):
            self.triggered = True
            return self.DEFECT

        return self.COOP

class Pavlov(Strategy):
    def act(self, state):
        if not state.history:
            return self.COOP

        my_last, opp_last, r_self, _ = state.history[-1]

        # if last outcome was "good" → repeat
        if r_self >= 2:  # CC or DC
            return my_last
        else:
            return 1 - my_last

In [5]:
# state = MatchState(N=5, T=100)
# games = state.match_play(TitForTwoTats(), Pavlov())

# print(games)

### Axelrods Strategies

In [6]:
class K59R(Strategy):
    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        # First move
        if not state.history:
            return COOP

        # Counters
        tot_cop = 0
        tot_def = 0
        nice1 = 0  # opponent cooperated after our cooperation
        nice2 = 0  # opponent cooperated after our defection

        # Build statistics
        for my_move, opp_move, _, _ in state.history:
            if my_move == COOP:
                tot_cop += 1
                if opp_move == COOP:
                    nice1 += 1
            else:
                tot_def += 1
                if opp_move == COOP:
                    nice2 += 1

        # Avoid division by zero
        good = (nice1 / tot_cop) if tot_cop > 0 else 1.0
        bad = (nice2 / tot_def) if tot_def > 0 else 0.0

        # Compute decision scores
        C = 6.0 * good - 8.0 * bad - 2.0
        ALT = 4.0 * good - 5.0 * bad - 1.0

        # Previous move (needed for flip behavior)
        prev_move = state.history[-1][0]

        # Decision logic
        if C >= 0.0 and C >= ALT:
            return COOP
        elif C >= 0.0 and C < ALT:
            return 1 - prev_move
        elif ALT >= 0.0:
            return 1 - prev_move
        else:
            return DEFECT
        
class K73R(Strategy):
    def reset(self):
        # Initialize variables (equivalent to M <= 1 case)
        self.IAGGD = 4
        self.IDUNU = 0
        self.IDUNB = 0
        self.IPAYB = 8
        self.ITEST = 1
        self.IPOST = 0  # default move

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1  # current round

        # First move initialization (Fortran: IF (M .GT. 1) GOTO 10)
        if M == 1:
            self.reset()

        # Default action
        move = self.IPOST

        # If no history yet, just return
        if not state.history:
            return move

        # J = opponent last move
        J = state.history[-1][1]

        # If opponent move != ITEST → do nothing
        if J != self.ITEST:
            return move

        # Update counters
        if self.ITEST == 1:
            self.IDUNU += 1
        else:
            self.IDUNB += 1

        # Check thresholds
        if (self.IDUNU < self.IAGGD) and (self.IDUNB < self.IPAYB):
            return move

        # Reset counters
        self.IDUNU = 0
        self.IDUNB = 0

        # Update IPOST
        self.IPOST = 0
        if J == 1:
            self.IPOST = 1

        move = self.IPOST

        # Switch test mode
        self.ITEST = 0
        if self.IPOST == 0:
            self.ITEST = 1

        # Compute K (approx = cumulative reward)
        K = sum(r_self for _, _, r_self, _ in state.history)

        # Update thresholds
        if self.ITEST == 1:
            self.IAGGD = self.IAGGD - 3 + (K // M)
            if self.IAGGD <= 0:
                self.IAGGD = 1
        else:
            self.IPAYB = int(1.6667 * (self.IAGGD + 1))

        return move

class K74R(Strategy):
    def reset(self):
        # Core parameters
        self.ALPHA = 1.0
        self.BETA = 0.3

        # Previous action
        self.IOLD = 0

        # Counters (with decay)
        self.QCA = 0.0
        self.QNA = 0.0
        self.QCB = 0.0
        self.QNB = 0.0

        # Output state
        self.K74R_val = 0

        # Random detection variables
        self.JSW = 0
        self.JS4 = 0
        self.JS11 = 0
        self.JR = 0
        self.JL = 0
        self.JT = 0
        self.JSM = 1

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        # Initialization
        if M == 1:
            self.reset()
            return COOP

        # If randomness detected → always defect
        if self.JR == 1:
            return DEFECT

        # Get opponent last move
        J = state.history[-1][1]

        # === Update ALPHA / BETA (after round 2) ===
        if M > 2:
            if self.IOLD == COOP:
                if J == COOP:
                    self.QCA += 1
                self.QNA += 1
                self.ALPHA = self.QCA / self.QNA if self.QNA > 0 else 1.0

                # decay
                self.QCA *= 0.8
                self.QNA *= 0.8
            else:
                if J == COOP:
                    self.QCB += 1
                self.QNB += 1
                self.BETA = self.QCB / self.QNB if self.QNB > 0 else 0.0

                # decay
                self.QCB *= 0.8
                self.QNB *= 0.8

        # Store previous move
        self.IOLD = self.K74R_val

        # === Randomness detection (before round 37) ===
        if M != 37:
            if M <= 37:
                if J == self.JL:
                    self.JSM += 1
                    if self.JSM >= 3:
                        self.JS4 = 1
                    if self.JSM >= 11:
                        self.JS11 = 1
                else:
                    self.JSW += 1
                    self.JSM = 1

                self.JT += J

        # Update last opponent move
        self.JL = J

        # === Policy computation ===
        POLC = 6 * self.ALPHA - 8 * self.BETA - 2
        POLALT = 4 * self.ALPHA - 5 * self.BETA - 1

        # Decision logic
        if POLC == 0:
            if POLC >= POLALT:
                self.K74R_val = COOP
                return COOP

        if POLALT >= 0:
            self.K74R_val = 1 - self.K74R_val
            return self.K74R_val

        # Otherwise
        if POLALT < 0:
            self.K74R_val = DEFECT
            return DEFECT

        # Fallback
        self.K74R_val = COOP
        return COOP

        # === Special check at round 37 ===
        if M == 37:
            if (self.JS4 == 1 and
                self.JS11 == 0 and
                10 < self.JT < 26 and
                self.JSW < 26):
                self.JR = 1


class K74RXX(Strategy):
    def reset(self):
        # Core parameters
        self.ALPHA = 1.0
        self.BETA = 0.3

        # Previous internal action
        self.IOLD = 0

        # Counters (with decay)
        self.QCA = 0.0
        self.QNA = 0.0
        self.QCB = 0.0
        self.QNB = 0.0

        # Internal action memory (IMPORTANT)
        self.k74dummy = 0

        # Random detection variables
        self.JSW = 0
        self.JS4 = 0
        self.JS11 = 0
        self.JR = 0
        self.JL = 0
        self.JT = 0
        self.JSM = 1

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        # === Initialization ===
        if M == 1:
            self.reset()
            return COOP

        # If randomness detected → always defect
        if self.JR == 1:
            self.k74dummy = DEFECT
            return DEFECT

        # Opponent last move
        J = state.history[-1][1]

        # === Update ALPHA / BETA ===
        if M > 2:
            if self.IOLD == COOP:
                if J == COOP:
                    self.QCA += 1
                self.QNA += 1
                self.ALPHA = self.QCA / self.QNA if self.QNA > 0 else 1.0

                # decay
                self.QCA *= 0.8
                self.QNA *= 0.8
            else:
                if J == COOP:
                    self.QCB += 1
                self.QNB += 1
                self.BETA = self.QCB / self.QNB if self.QNB > 0 else 0.0

                # decay
                self.QCB *= 0.8
                self.QNB *= 0.8

        # Store previous internal move
        self.IOLD = self.k74dummy

        # === Randomness detection (pre-37 rounds) ===
        if M != 37:
            if M <= 37:
                if J == self.JL:
                    self.JSM += 1
                    if self.JSM >= 3:
                        self.JS4 = 1
                    if self.JSM >= 11:
                        self.JS11 = 1
                else:
                    self.JSW += 1
                    self.JSM = 1

                self.JT += J

        # Update last opponent move
        self.JL = J

        # === Policy computation ===
        POLC = 6 * self.ALPHA - 8 * self.BETA - 2
        POLALT = 4 * self.ALPHA - 5 * self.BETA - 1

        # === Decision logic ===
        if POLC == 0:
            if POLC >= POLALT:
                self.k74dummy = COOP
                return COOP

        if POLALT >= 0:
            self.k74dummy = 1 - self.k74dummy
            return self.k74dummy

        # Otherwise defect
        self.k74dummy = DEFECT
        return DEFECT

        # === Round 37 special detection ===
        if M == 37:
            if (self.JS4 == 1 and
                self.JS11 == 0 and
                10 < self.JT < 26 and
                self.JSW < 26):
                self.JR = 1

class K75R(Strategy):
    def reset(self):
        # HIST[4][2]
        self.HIST = [[0, 0] for _ in range(4)]

        self.IBURN = 0
        self.ID = [0, 0]
        self.IDEF = 0
        self.ITWIN = 0
        self.ISTRNG = 0
        self.ICOOP = 0
        self.ITRY = 0
        self.IRDCHK = 0
        self.IRAND = 0
        self.IPARTY = 1
        self.IND = 0
        self.MY = 0
        self.INDEF = 5
        self.IOPP = 0
        self.PROB = 0.2

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        # === Initialization ===
        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]  # opponent last move

        # === Random mode handling ===
        if self.IRAND == 1:
            self.IRDCHK += J * 4 - 3
            if self.IRDCHK < 11:
                self.MY = DEFECT
                return DEFECT
            else:
                self.IRAND = 2
                self.ICOOP = 2
                self.MY = COOP
                return COOP

        # === Update stats ===
        self.IOPP += J
        self.HIST[self.IND][J] += 1

        # === Randomness detection every 15 rounds ===
        if (M >= 15 and M % 15 == 0 and self.IRAND != 2):
            if self.HIST[0][0] / max(1, (M - 2)) < 0.8:
                if (self.IOPP * 4 >= (M - 2)) and (self.IOPP * 4 <= 3 * M - 6):

                    ROW = [sum(row) for row in self.HIST]
                    COL = [sum(self.HIST[i][j] for i in range(4)) for j in range(2)]

                    chi = 0.0
                    for i in range(4):
                        for j2 in range(2):
                            expected = ROW[i] * COL[j2] / max(1, (M - 2))
                            if expected > 1:
                                diff = self.HIST[i][j2] - expected
                                chi += (diff ** 2) / expected

                    if chi <= 3:
                        self.IRAND = 1
                        self.MY = DEFECT
                        return DEFECT

        # === Behavior logic ===

        if self.ITRY == 1 and J == DEFECT:
            self.IBURN = 1

        if M <= 37 and J == COOP:
            self.ITWIN += 1

        if M == 38 and J == DEFECT:
            self.ITWIN += 1

        if M >= 39 and self.ITWIN == 37 and J == DEFECT:
            self.ITWIN = 0

        if self.ITWIN == 37:
            return self._coop_phase()

        self.IDEF = self.IDEF * J + J
        if self.IDEF >= 20:
            return self._defect_phase()

        self.IPARTY = 3 - self.IPARTY
        idx = self.IPARTY - 1

        self.ID[idx] = self.ID[idx] * J + J
        if self.ID[idx] >= self.INDEF:
            self.ID[idx] = 0
            self.ISTRNG += 1
            if self.ISTRNG == 8:
                self.INDEF = 3

        if self.ICOOP >= 1:
            return self._coop_phase()

        if M >= 37 and self.IBURN == 0:
            if M == 37 or (state.history[-1][2] <= self.PROB):
                self.ITRY = 2
                self.ICOOP = 2
                self.PROB += 0.05
                return self._defect_phase()

        if J == COOP:
            return self._coop_phase()

        return self._defect_phase()

    # === Helper phases ===
    def _coop_phase(self):
        self.ITRY -= 1
        self.ICOOP -= 1
        self.MY = self.COOP
        self._update_index()
        return self.COOP

    def _defect_phase(self):
        idx = self.IPARTY - 1
        self.ID[idx] += 1
        self.MY = self.DEFECT
        self._update_index()
        return self.DEFECT

    def _update_index(self):
        # IND = 2 * MY + J
        # J is last opponent move
        # Need safe fallback if no history
        if hasattr(self, "_last_J"):
            J = self._last_J
        else:
            J = 0
        self.IND = 2 * self.MY + J
        self._last_J = J

In [7]:
#Tit for tat with no memory (only last move)
class K92R(Strategy): 
    def act(self, state):
        # First move → cooperate
        if not state.history:
            return self.COOP

        # Copy opponent's last move (Tit For Tat)
        return state.history[-1][1]
    
class KPavlovC(Strategy):
    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        # First move → cooperate
        if not state.history:
            return COOP

        my_last, opp_last, _, _ = state.history[-1]

        # If same move → cooperate, else defect
        if opp_last == my_last:
            return COOP
        else:
            return DEFECT

class KRandomC(Strategy):
    def act(self, state):
        # 50% chance defect, 50% cooperate
        if random.random() <= 0.5:
            return self.DEFECT
        return self.COOP

class KTF2TC(Strategy):
    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        # First or second move → cooperate
        if len(state.history) < 2:
            return COOP

        # Check last two opponent moves
        opp_last = state.history[-1][1]
        opp_prev = state.history[-2][1]

        if opp_last == DEFECT and opp_prev == DEFECT:
            return DEFECT

        return COOP
    
class KTitForTatC(Strategy):
    def act(self, state):
        # First move → cooperate
        if not state.history:
            return self.COOP

        # Tit For Tat
        return state.history[-1][1]

In [8]:
class GRASR(Strategy):
    def reset(self):
        self.NMOV = [0, 0, 0, 0]
        self.NMOVE = 0
        self.IGAME = 0
        self.N = 0
        self.MMOVE = 0

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        # Initialization
        if M == 1:
            self.reset()
            return COOP

        # JPICK = opponent last move
        JPICK = state.history[-1][1]

        # Approximate JSCOR (opponent cumulative reward)
        JSCOR = sum(r_opp for _, _, _, r_opp in state.history)

        # Random number
        RANDO = random.random()

        # MOVEN <= 2
        if M <= 2:
            return COOP

        # MOVEN < 51
        if M < 51:
            return JPICK

        # MOVEN == 51
        if M == 51:
            return DEFECT

        # 52 <= MOVEN < 57
        if M < 57:
            idx = M - 53
            if 0 <= idx < 4:
                self.NMOV[idx] = self.MMOVE + JPICK

            move = JPICK

            if move == COOP:
                self.MMOVE = 2
            else:
                self.MMOVE = 4

            return move

        # MOVEN == 57
        if M == 57:
            if JSCOR > 135:
                J = self.NMOV[1]

                if J == 1:
                    if self.NMOV[0] == 3 and self.NMOV[2] >= 3:
                        self.IGAME = 2
                        return COOP

                elif J == 2:
                    if self.NMOV[0] == 2 and self.NMOV[2] >= 4 and self.NMOV[3] >= 2:
                        self.IGAME = 4
                        return COOP

                elif J == 3:
                    if self.NMOV[0] == 5 and self.NMOV[2] >= 5:
                        self.IGAME = 2
                        return COOP

                self.IGAME = 1
                self.N = int(RANDO * 10 + 5)
                return COOP
            else:
                self.IGAME = 3
                return DEFECT

        # MOVEN > 57
        if self.IGAME == 1:
            if self.N <= 0:
                self.N = int(RANDO * 10 + 5)
                return DEFECT
            else:
                self.N -= 1
                return JPICK

        elif self.IGAME == 2:
            return JPICK

        elif self.IGAME == 3:
            return DEFECT

        elif self.IGAME == 4:
            if M >= 118:
                self.IGAME = 2
            return COOP

        return COOP

class K31R(Strategy):
    def reset(self):
        self.S = 0.0

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return DEFECT  # since A = 0 < 0.5

        # J = opponent last move
        J = state.history[-1][1]

        self.S += J
        A = self.S / M

        if A < 0.5:
            return COOP
        return DEFECT

class K32R(Strategy):
    def reset(self):
        self.C1 = 0  # opp coop after my defect
        self.C2 = 0  # opp defect after my defect
        self.C3 = 0  # opp coop after my coop
        self.C4 = 0  # opp defect after my coop

        self.J2 = 0
        self.J1 = 0
        self.I2 = 0
        self.I1 = 0

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]
        R = random.random()

        # === Update statistics (from M >= 3) ===
        if M > 2:
            if self.I2 == DEFECT:
                if J == COOP:
                    self.C1 += 1
                else:
                    self.C2 += 1
            else:
                if J == COOP:
                    self.C3 += 1
                else:
                    self.C4 += 1

            # Random detection (after move 26)
            if M >= 27:
                if (self.C1 >= ((self.C1 + self.C2) - 1.5 * math.sqrt(self.C1 + self.C2)) / 2 and
                    self.C4 >= ((self.C3 + self.C4) - 1.5 * math.sqrt(self.C3 + self.C4)) / 2):
                    move = DEFECT
                    self._update_memory(J, move)
                    return move

        # === Decision logic ===
        move = COOP

        if self.J1 == J:
            if self.J2 == self.J1:
                move = J  # 3 in a row
            else:
                P = 0.9  # 2 in a row
                move = J if R < P else 1 - J
        else:
            if J == DEFECT:
                P = 0.6
            else:
                P = 0.7
            move = J if R < P else 1 - J

        self._update_memory(J, move)
        return move

    def _update_memory(self, J, move):
        self.J2 = self.J1
        self.J1 = J
        self.I2 = self.I1
        self.I1 = move

In [9]:
class K33R(Strategy):
    def reset(self):
        # stats (rename to avoid collision)
        self.coop_count = [0.0] * 4
        self.count = [0.0] * 4
        self.P = [0.0] * 4

        # history of own moves
        self.LAST1 = 1
        self.LAST2 = 1

        self.TWIN = True

        # constants
        self.CONST = [0., 4., 6., 6., 8., 12.]
        self.COEFF = [
            [36., 16., 0., 12.],
            [0., 12., 18., 12.],
            [0., 12., 24., 9.],
            [0., 0., 0., 9.],
            [12., 48., 0., 0.],
            [0., 0., 0., 0.]
        ]

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]

        INDEX = 2 * self.LAST2 + self.LAST1

        # update probability estimates
        if M > 2:
            self.coop_count[INDEX] += (1 - J)
            self.count[INDEX] += 1
            if self.count[INDEX] > 0:
                self.P[INDEX] = self.coop_count[INDEX] / self.count[INDEX]

        INDEX = 2 * self.LAST2 + self.LAST1

        if M > 1 and J != self.LAST1:
            self.TWIN = False

        if M <= 22:
            move = COOP if INDEX in [0, 1] else DEFECT
            self._update(move)
            return int(move)

        if self.TWIN:
            move = COOP
            self._update(move)
            return int(move)

        BEST = -float("inf")
        IPOL = 0

        for i in range(6):
            SUM = self.CONST[i]
            for j in range(4):
                SUM += self.COEFF[i][j] * self.P[j]
            if SUM > BEST:
                BEST = SUM
                IPOL = i

        policy_map = {
            0: [COOP, COOP, COOP, COOP],
            1: [DEFECT, COOP, COOP, COOP],
            2: [DEFECT, COOP, DEFECT, COOP],
            3: [DEFECT, DEFECT, COOP, COOP],
            4: [DEFECT, DEFECT, DEFECT, COOP],
            5: [DEFECT, DEFECT, DEFECT, DEFECT],
        }

        policy = policy_map.get(IPOL, [COOP] * 4)
        move = int(policy[INDEX])

        self._update(move)
        return move

    def _update(self, move):
        self.LAST2 = self.LAST1
        self.LAST1 = move

class K34R(Strategy):
    def reset(self):
        self.JT = 0

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]

        self.JT += J

        if self.JT > 0:
            return DEFECT
        return COOP

import random

class K35R(Strategy):
    def reset(self):
        self.FLACK = 0.0

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]
        R = random.random()

        # Update flack (exponential decay with factor 0.5)
        self.FLACK = (self.FLACK + J) * 0.5

        if self.FLACK > R:
            return DEFECT
        return COOP

class K36R(Strategy):
    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1
        R = random.random()

        # Default: DEFECT
        move = DEFECT

        if 1 <= M < 100:
            PROBC = 0.1
        elif 100 <= M < 200:
            PROBC = 0.05
        elif 200 <= M < 300:
            PROBC = 0.15
        else:
            PROBC = 0.0

        if R < PROBC:
            move = COOP

        return move

In [10]:
class K37R(Strategy):
    def reset(self):
        self.ND = 0  # number of opponent defections

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]

        self.ND += J

        if 5 * self.ND > M:
            return DEFECT
        return COOP

class K38R(Strategy):
    def reset(self):
        self.MOVE = 0
        self.JHIS = 0  # stores last 3 opponent moves as bit pattern

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]

        if self.MOVE != DEFECT:
            if self.JHIS >= 4:
                self.JHIS -= 4
            self.JHIS = self.JHIS * 2 + J

            if self.JHIS == 7:  # 111 → three consecutive defections
                self.MOVE = DEFECT

        return self.MOVE

class K39R(Strategy):
    def reset(self):
        self.STEP = 1
        self.SUBSTP = 1
        self.BOTHD = 0
        self.TITCNT = 0
        self.TATCNT = 0
        self.EVIL = 0
        self.N = 1
        self.F = 0
        self.OK = [0, 0, 0]
        self.TOTK = 0
        self.OLDMOV = 0
        self.COUNT = 0

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]
        K = sum(r_self for _, _, r_self, _ in state.history)

        # --- tabulation ---
        prev_move = getattr(self, "last_move", COOP)

        if prev_move + J == 2:
            self.BOTHD += 1
        else:
            self.BOTHD = 0

        self.COUNT -= 1

        move = COOP

        VOLDMV = self.OLDMOV
        self.OLDMOV = J

        if J == DEFECT:
            self.TATCNT += 1
        if self.EVIL == 0 and J == DEFECT:
            self.EVIL = 1

        # --- main FSM ---
        while True:
            if self.STEP == 1:
                # Tit for Two Tats mode
                if self.SUBSTP == 1:
                    self.COUNT = 10
                    self.TATCNT = 0
                    self.TITCNT = 0
                    self.SUBSTP = 2
                    continue

                elif self.SUBSTP == 2:
                    if (VOLDMV + self.OLDMOV) == 2:
                        move = DEFECT
                    else:
                        move = COOP

                    self.TITCNT += move

                    if self.COUNT == 0:
                        self.SUBSTP = 3
                    break

                elif self.SUBSTP == 3:
                    OLDSTP = self.STEP
                    self.OK[self.STEP - 1] = K - self.TOTK
                    self.TOTK = K
                    self.SUBSTP = 1

                    if self.TATCNT == 0:
                        self.STEP = 4
                        if self.EVIL == 1:
                            self.STEP = 1
                        if self.EVIL == 0:
                            self.EVIL = -1
                        continue

                    self.STEP = 1
                    for i1 in range(2):
                        for i2 in range(1, 3):
                            if self.OK[i1] == 0 or self.OK[i2] == 0:
                                continue
                            if self.OK[i1] < self.OK[i2]:
                                if self.STEP == i1 + 1:
                                    self.STEP = i2 + 1

                    if self.STEP != 3:
                        if (self.OK[min(self.STEP, 2)] == 0 and
                            (self.TATCNT >= 4 or self.TITCNT == 0)):
                            self.STEP += 1

                    if self.STEP < OLDSTP and self.BOTHD > 0:
                        self.STEP = 5

                    continue

            elif self.STEP == 2:
                # Tit for Tat
                if self.SUBSTP == 1:
                    self.COUNT = 10
                    self.SUBSTP = 2
                    continue

                elif self.SUBSTP == 2:
                    move = DEFECT if self.OLDMOV == DEFECT else COOP
                    self.TITCNT += move
                    if self.COUNT == 0:
                        self.SUBSTP = 3
                    break

                elif self.SUBSTP == 3:
                    self.SUBSTP = 1
                    self.STEP = 1
                    continue

            elif self.STEP == 3:
                # Always defect
                if self.SUBSTP == 1:
                    self.COUNT = 10
                    self.SUBSTP = 2
                    continue

                elif self.SUBSTP == 2:
                    move = DEFECT
                    self.TITCNT += 1
                    if self.COUNT == 0:
                        self.SUBSTP = 3
                    break

                elif self.SUBSTP == 3:
                    self.SUBSTP = 1
                    self.STEP = 1
                    continue

            elif self.STEP == 4:
                # Exploit
                if self.SUBSTP == 1:
                    self.SUBSTP = 2
                    move = DEFECT
                    self.COUNT = self.N
                    self.TATCNT = 0
                    break

                elif self.SUBSTP == 2:
                    if self.COUNT == 0:
                        self.SUBSTP = 3
                    break

                elif self.SUBSTP == 3:
                    if self.TATCNT == 0:
                        self.F = 1
                        self.SUBSTP = 1
                        continue
                    else:
                        if self.F == 1:
                            self.N += 1
                            self.SUBSTP = 1
                            self.STEP = 1
                            continue
                        else:
                            self.SUBSTP = 4
                            if J == DEFECT:
                                self.N += 1
                            self.TATCNT = J
                            break

                elif self.SUBSTP == 4:
                    if self.TATCNT > 4:
                        self.SUBSTP = 1
                        self.STEP = 1
                        continue
                    break

            elif self.STEP == 5:
                # Cooling cooperation
                if self.SUBSTP != 2:
                    self.COUNT = 5
                    self.SUBSTP = 2

                if self.COUNT != 0:
                    break
                else:
                    self.SUBSTP = 1
                    self.STEP = 1
                    continue

            break

        self.last_move = move
        return move

class K40R(Strategy):
    def reset(self):
        self.S = 3
        self.W = 0
        self.Q = 0.8

    def act(self, state):
        COOP = self.COOP
        DEFECT = self.DEFECT

        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return COOP

        J = state.history[-1][1]
        R = random.random()

        self.S += 1

        if J == DEFECT:
            self.W += 1
            self.Q *= 0.5

        if M < 3:
            return COOP

        if J == DEFECT:
            self.W += 1

            cond1 = (self.W > 2 and (self.W / 3 == int(self.W / 3)))
            cond2 = ((self.W - 1) / 3 == int((self.W - 1) / 3))

            if cond1 or cond2:
                self.S = 1
                self.Q *= 0.5
            else:
                if R < self.Q:
                    self.Q *= 0.5
                    return COOP
                else:
                    self.Q *= 0.5
                    return DEFECT
        # else J == COOP → go to 580

        if self.S in [1, 2]:
            return DEFECT

        cond1 = (self.W > 2 and (self.W / 3 == int(self.W / 3)))
        cond2 = ((self.W - 1) / 3 == int((self.W - 1) / 3))

        if cond1 or cond2:
            self.S = 1
            self.Q *= 0.5
            return DEFECT

        return COOP

In [11]:
class K41R(Strategy):
    def reset(self):
        self.LAST = [0] * 12
        self.ICASE = 1
        self.IFORGV = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else self.COOP
        M = len(state.history) + 1  # assuming M is round index (1-based)

        # Initialize on first move
        if M == 1:
            self.ICASE = 1
            self.IFORGV = 0
            self.LAST = [0] * 12

        # Main logic
        if self.ICASE == 1:
            K41R = J
            self.ICASE = J + 1

        elif self.ICASE == 2:
            K41R = J
            self.ICASE = 3
            if J == 1:
                self.ICASE = 1

        elif self.ICASE == 3:
            K41R = J
            if self.IFORGV < M:
                K41R = 0
            self.IFORGV = self.IFORGV + 20 * J
            self.ICASE = 1

        # Update LAST and compute sum
        LSUM = self.LAST[0]
        for i in range(1, 12):
            LSUM += self.LAST[i]
            self.LAST[i - 1] = self.LAST[i]

        self.LAST[11] = J

        if LSUM >= 5:
            K41R = 1

        return K41R

class K42R(Strategy):
    def reset(self):
        self.MHIST = [[0, 0], [0, 0]]
        self.L3MOV = 0
        self.L3ECH = 0
        self.IDEF = 0
        self.ICOOP = 0
        self.IPICK = 0
        self.I2PCK = 0
        self.J2PCK = 0

    def act(self, state):
        JPICK = state.history[-1][1] if state.history else 0
        MOVEN = len(state.history) + 1

        # First move initialization
        if MOVEN == 1:
            self.L3MOV = 0
            self.L3ECH = 0
            self.IDEF = 0
            self.ICOOP = 0
            self.IPICK = 0
            self.I2PCK = 0
            self.J2PCK = 0
            self.MHIST = [[0, 0], [0, 0]]
            return self.COOP

        # Update move history
        if MOVEN != 2:
            self.MHIST[self.I2PCK][JPICK] += 1

        # If opponent marked defective/random → defect
        if self.IDEF != 0:
            K42R = self.DEFECT
        else:
            # Mutual defection
            if self.IPICK == 1 and JPICK == 1:
                self.L3MOV += 1
                if self.L3MOV >= 3:
                    K42R = self.COOP
                    self.L3MOV = 0
                    self.L3ECH = 0
                else:
                    K42R = JPICK
            else:
                self.L3MOV = 0

                # Echo effect
                if (
                    self.IPICK != JPICK and
                    JPICK == self.I2PCK and
                    self.IPICK == self.J2PCK
                ):
                    self.L3ECH += 1
                    if self.L3ECH >= 3:
                        self.L3ECH = 0
                        self.L3MOV = 0
                        self.ICOOP = 1
                else:
                    self.L3ECH = 0

                K42R = JPICK  # Tit for Tat

        # Every 25 moves check opponent type
        if (MOVEN - 2) % 25 == 0 and MOVEN != 2:
            self.IDEF = 0
            JNCOP = self.MHIST[0][0] + self.MHIST[1][0]

            # Random check
            if JNCOP > 17:
                pass
            elif JNCOP < 8:
                if JNCOP < 3:
                    self.IDEF = 1
            else:
                if 100 * self.MHIST[0][0] / max(JNCOP, 1) < 70:
                    self.IDEF = 1

            # Reset history
            self.MHIST = [[0, 0], [0, 0]]

            if self.IDEF != 0:
                self.ICOOP = 0
                self.L3MOV = 0
                self.L3ECH = 0
                K42R = self.DEFECT

        # Forced cooperation override
        if self.ICOOP != 0 and K42R == self.DEFECT:
            self.ICOOP = 0
            K42R = self.COOP

        # Update picks
        self.I2PCK = self.IPICK
        self.J2PCK = JPICK
        self.IPICK = K42R

        return K42R

class K43R(Strategy):
    def reset(self):
        self.NCC = 0
        self.NCD = 0
        self.NDC = 0
        self.NDD = 0
        self.KOUNT = 0
        self.MYTWIN = 0
        self.IOLD1 = 0
        self.IOLD2 = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M <= 1:
            self.NCC = 0
            self.NCD = 0
            self.NDC = 0
            self.NDD = 0
            self.KOUNT = 0
            self.MYTWIN = 0
            self.IOLD1 = 0
            return self.IOLD1

        if M >= 3:
            if self.IOLD2 == 1:
                self.NDC += (1 - J)
                self.NDD += J
            else:
                self.NCC += (1 - J)
                self.NCD += J

        self.IOLD2 = self.IOLD1

        if M < 16:
            if J == 0 or self.KOUNT >= 3:
                self.IOLD1 = 0
                return self.IOLD1
            self.KOUNT += 1
            self.IOLD1 = 1
            return self.IOLD1

        if M == 17 and J == 1 and self.NCD == 1 and self.NDD == 0:
            self.MYTWIN = 1

        if (self.NCD * 3) >= (self.NCC + self.NCD):
            self.IOLD1 = 1
            return self.IOLD1

        if M % 4 != 0:
            self.IOLD1 = 0
            return self.IOLD1

        if self.MYTWIN == 1:
            self.IOLD1 = 0
            return self.IOLD1

        if self.NDC >= (M // 12) or self.NDD == 0:
            self.IOLD1 = 1
            return self.IOLD1

        self.IOLD1 = 0
        return self.IOLD1

class K44R(Strategy):
    def reset(self):
        self.MC = 0
        self.F = 2
        self.AM = 4

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        # Initialization on first move
        if M == 1:
            self.MC = 0
            self.F = 2
            self.AM = 4
            return self.COOP

        if M < 3:
            return self.COOP

        # Count opponent defects
        self.MC += J

        # Cooperate until threshold
        if self.MC < self.AM:
            return self.COOP

        if self.MC == self.AM:
            return self.DEFECT

        # Adjust threshold
        self.AM = self.AM / self.F
        self.MC = 0

        # Probabilistic forgiveness
        if R < self.AM:
            return self.COOP

        return self.DEFECT

class K45R(Strategy):
    def reset(self):
        self.JOLD = 0
        self.A = 0
        self.B = 0
        self.C = 0
        self.D = 0
        self.E = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M <= 3:
            if M == 1:
                self.JOLD = 0
                self.A = 0
                self.B = 0
                self.C = 0
                self.E = 0
                return self.DEFECT

            if M == 2:
                if J == 1:
                    self.D = 1
                else:
                    self.D = 0
                return self.COOP

            # M == 3
            if J == 1:
                if self.D == 1:
                    self.C = 1
                return self.COOP
            else:
                if self.D == 1:
                    return self.COOP
                self.A = 1
                return self.COOP

        # M > 3
        if self.C == 1:
            return J

        if self.B == 1:
            K45R = self.COOP
            if self.JOLD == 1 and J == 1:
                K45R = self.DEFECT
            self.JOLD = J
            return K45R

        if self.A == 1:
            K45R = self.DEFECT
            self.E += 1
            if self.E == 8:
                self.E = 0
                self.JOLD = J
                return K45R
            if not (self.JOLD == 1 and J == 1):
                K45R = self.COOP
            self.JOLD = J
            return K45R

        if self.D == 1:
            if J == 1:
                self.C = 1
                return self.DEFECT
            else:
                self.B = 1
                return self.COOP

        if J == 1:
            self.C = 1
            return self.COOP
        else:
            self.B = 1
            return self.COOP

In [12]:
class K46R(Strategy):
    def reset(self):
        self.NJ = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if M == 1:
            self.NJ = 0

        self.NJ += J

        K46R = self.COOP

        if J == 0:
            return K46R

        if M - 1 > 0:
            P = float(self.NJ) / float(M - 1)
        else:
            P = 0.0

        if R < P:
            K46R = self.DEFECT

        return K46R
    
class K47R(Strategy):
    def reset(self):
        self.NUM = 2
        self.DEN = 2
        self.RF = 20
        self.DEF = self.DEFECT
        self.COOP = self.COOP
        self.LONG = 1
        self.SHORT = 5
        self.SH2 = [1] * 5
        self.N = 1
        self.MYLAST = 0
        self.MYMOVE = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.reset()

        # Update RF if early defection
        if M <= self.RF and J == self.DEF:
            self.RF = M + (20 * self.NUM) // self.DEN + 1

        # Update index
        self.N = (self.N % 4) + 1

        # Update SHORT
        self.SHORT -= self.SH2[self.N - 1]

        if J == self.MYLAST:
            self.LONG += 1
            self.SHORT += 1
            self.SH2[self.N - 1] = 1
        else:
            self.SH2[self.N - 1] = 0

        self.MYLAST = self.MYMOVE

        # Default move
        self.MYMOVE = J

        if (self.LONG < 0.625 * M) or (self.SHORT < 3):
            self.MYMOVE = self.DEF

        if (self.LONG > 0.9 * M) and (self.SHORT == 5):
            self.MYMOVE = self.COOP

        # Forced retaliation (RF)
        if M == self.RF:
            self.MYMOVE = self.DEF

        if M >= self.RF + 2:
            self.MYMOVE = self.COOP
            self.NUM += J
            self.DEN += (1 - J)
            if self.DEN != 0:
                self.RF = M + (20 * self.NUM) // self.DEN + 1

        return self.MYMOVE
    
class K48R(Strategy):
    def reset(self):
        self.IARRAY = [0] * 5
        self.IPO2 = [2, 4, 3, 5, 1]  # 1-based in Fortran
        self.KOLD = 0
        self.K5 = 0
        self.KLAST = 0
        self.MM = 0
        self.ICHAN = 1
        self.IPO1 = 1  # 1-based

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        K = getattr(state, "score", 0)

        if M == 1:
            self.KOLD = 0
            self.K5 = 0
            self.KLAST = 0
            self.IARRAY = [0] * 5
            self.MM = 0
            self.ICHAN = 1
            self.IPO1 = 1

            idx = self.IPO2[self.IPO1 - 1] - 1
            self.IARRAY[idx] = J
            self.IPO1 += J
            return J

        if M <= 5:
            idx = self.IPO2[self.IPO1 - 1] - 1
            self.IARRAY[idx] = J
            self.IPO1 += J
            return J

        self.MM = ((M - 1) % 5) + 1
        K48R = self.IARRAY[self.MM - 1]

        if self.MM != 1:
            return K48R

        KOLD = self.K5
        self.K5 = K - self.KLAST
        self.KLAST = K

        if KOLD > self.K5:
            self.ICHAN = -self.ICHAN
            self.IPO1 += self.ICHAN

        if self.IPO1 < 1:
            self.IPO1 = 0
        if self.IPO1 > 5:
            self.IPO1 = 6

        if self.IPO1 < 1 or self.IPO1 > 5:
            return K48R

        idx = self.IPO2[self.IPO1 - 1] - 1
        self.IARRAY[idx] += self.ICHAN
        self.IPO1 += self.ICHAN

        return self.IARRAY[self.MM - 1]
    
class K49R(Strategy):
    def reset(self):
        self.JDSUM = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if M == 1:
            self.JDSUM = 0

        # Count opponent defections
        if J == 1:
            self.JDSUM += 1

        # Percentage of defections
        JDPC = (100 * self.JDSUM) // M if M > 0 else 0

        # Base behavior
        if J == 0:
            K49R = self.COOP
        elif J == 1 and self.JDSUM <= 17:
            K49R = int(R + 0.5)
        else:
            K49R = self.DEFECT

        # Give up if too defective/random
        if (M > 19 and JDPC > 79) or \
           (M > 29 and JDPC > 65) or \
           (M > 39 and JDPC > 39):
            K49R = self.DEFECT

        return K49R
    
class K50R(Strategy):
    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        R = state.random if hasattr(state, "random") else 0.0

        K50R = self.COOP
        if J == 0 and R >= 0.9:
            K50R = self.DEFECT

        return K50R

In [13]:
class K51R(Strategy):
    def reset(self):
        self.LASTI = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M <= 8:
            K51R = self.COOP
            if M == 6:
                K51R = self.DEFECT
            self.LASTI = 0
            return K51R

        K51R = self.COOP
        self.LASTI -= 1

        if self.LASTI == 3:
            K51R = self.DEFECT

        if self.LASTI > 0:
            return K51R

        if J == 1:
            K51R = self.DEFECT
            self.LASTI = 4

        return K51R
    
class K52R(Strategy):
    def reset(self):
        self.D8 = 0
        self.D9 = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if M == 1:
            self.D8 = 0
            self.D9 = 0

        K52R = self.COOP

        self.D9 += 1
        if J <= 0:
            self.D9 = 0

        if self.D9 >= 2:
            K52R = self.DEFECT
            if self.D9 >= (5 + 3 * self.D8):
                self.D9 = 0
                self.D8 += 1

        if R <= 0.05:
            K52R = 1 - K52R

        return K52R
    
class K53R(Strategy):
    def reset(self):
        self.C = [0] * 10  # last 10 opponent moves

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        # Fill initial history
        if M <= 10:
            self.C[M - 1] = J
            return self.COOP

        # Shift history
        self.C = self.C[1:] + [J]

        # Count defections (J == 1)
        D = sum(1 for x in self.C if x == 1)

        # Decision rules
        if D > 8:
            if R < 0.94:
                return self.DEFECT
            return self.COOP

        if D == 8:
            if R < 0.915:
                return self.DEFECT
            return self.COOP

        if D in [7, 6, 5]:
            if R < 0.87:
                return self.DEFECT
            return self.COOP

        if D in [4, 3]:
            if R < 0.915:
                return self.DEFECT
            return self.COOP

        if D == 2:
            if R < 0.87:
                return self.DEFECT
            return self.COOP

        if D == 1:
            if R < 0.23:
                return self.DEFECT
            return self.COOP

        # D == 0
        return self.COOP
    
class K54R(Strategy):
    def reset(self):
        self.OPDEF = 0
        self.STDEF = 0
        self.DL = 0.20
        self.COOPS = 0
        self.OKDEF = True
        self.MYDEF = False
        self.NODEF = 0
        self.ND = 12

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        K54R = self.COOP

        if M == 1:
            self.OPDEF = 0
            self.STDEF = 0
            self.DL = 0.20
            self.COOPS = 0
            self.OKDEF = True
            self.MYDEF = False
            self.NODEF = 0
            self.ND = 12
            return K54R

        if M == 20:
            self.DL = 0.10

        # Opponent cooperates
        if J == 0:
            self.STDEF = 0
            self.COOPS += 1

            if float(self.OPDEF) > float(M) * self.DL:
                pass
            elif (M % self.ND == 0) and self.OKDEF:
                K54R = self.DEFECT
                self.MYDEF = False
                self.NODEF += 1
                if self.NODEF % 6 == 0:
                    self.ND -= 1
                if self.ND < 1:
                    self.ND = 1
                return K54R
            else:
                self.MYDEF = False
                return K54R

        # Opponent defects
        self.COOPS = 0

        if M <= 4:
            return self.DEFECT

        self.STDEF += 1
        self.OPDEF += 1

        if self.MYDEF:
            self.OKDEF = False

        if float(self.OPDEF) > float(M) * self.DL or self.STDEF > 2:
            if 20 * self.OPDEF <= self.COOPS * M:
                return K54R
            return self.DEFECT

        self.MYDEF = False
        return K54R

class K55R(Strategy):
    def reset(self):
        self.ALPHA = 1.0
        self.BETA = 0.0
        self.IOLD = 0
        self.QCA = 0
        self.QNA = 0
        self.QCB = 0
        self.QNB = 0
        self.MUTDEF = 0
        self.last_move = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.reset()

        # Update stats
        if M > 2:
            if self.IOLD == 1:
                if J == 0:
                    self.QCB += 1
                self.QNB += 1
                if self.QNB > 0:
                    self.BETA = self.QCB / self.QNB
            else:
                if J == 0:
                    self.QCA += 1
                self.QNA += 1
                if self.QNA > 0:
                    self.ALPHA = self.QCA / self.QNA

        # Compute policies
        POLC = 6 * self.ALPHA - 9 * self.BETA - 2
        POLALT = 4 * self.ALPHA - 6 * self.BETA - 1

        # Decision
        if POLC >= 0 and POLC >= POLALT:
            K55R = self.COOP

        elif POLC < 0 and POLALT < 0:
            K55R = self.DEFECT

            if J == 1 and self.IOLD == 1:
                self.MUTDEF += 1
                if self.MUTDEF > 3:
                    K55R = self.COOP
            else:
                self.MUTDEF = 0

        else:
            # Alternate
            K55R = 1 - self.last_move

        # Save own past move
        self.IOLD = K55R
        self.last_move = K55R

        return K55R

In [14]:
class K56R(Strategy):
    def reset(self):
        self.GOOD = 1.0
        self.BAD = 0.0
        self.PAST = 0
        self.TOTCOP = 0
        self.TOTDEF = 0
        self.NICE1 = 0
        self.NICE2 = 0
        self.COOP = self.COOP
        self.DEFECT = self.DEFECT
        self.last_move = 0

    def act(self, state):
        LASTMV = state.history[-1][1] if state.history else 0
        MOVEN = len(state.history) + 1

        if MOVEN == 1:
            self.reset()

        elif MOVEN > 2:
            if self.PAST == self.DEFECT:
                if LASTMV == self.COOP:
                    self.NICE2 += 1
                self.TOTDEF += 1
                if self.TOTDEF > 0:
                    self.BAD = self.NICE2 / self.TOTDEF
            else:
                if LASTMV == self.COOP:
                    self.NICE1 += 1
                self.TOTCOP += 1
                if self.TOTCOP > 0:
                    self.GOOD = self.NICE1 / self.TOTCOP

        # Compute policy values
        C = 6.0 * self.GOOD - 8.0 * self.BAD - 2.0
        ALT = 4.0 * self.GOOD - 5.0 * self.BAD - 1.0

        # Decision
        if C >= 0.0 and C >= ALT:
            K56R = self.COOP
        elif C >= 0.0 and C < ALT:
            K56R = 1 - self.last_move
        elif ALT >= 0.0:
            K56R = 1 - self.last_move
        else:
            K56R = self.DEFECT

        self.PAST = K56R
        self.last_move = K56R

        return K56R
class K57R(Strategy):
    def reset(self):
        self.n = 0
        self.last_move = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.n = 0
            self.last_move = self.COOP
            return self.COOP

        # update 3-move history encoding
        self.n = 4 * (self.n % 16) + 2 * self.last_move + J

        if M <= 3:
            K57R = J
            if M == 3 and self.n == 6:
                K57R = self.DEFECT
            self.last_move = K57R
            return K57R

        n = self.n
        K57R = self.DEFECT  # default

        if n < 39:
            if n <= 0:
                K57R = self.COOP
            elif n < 28:
                if n == 27:
                    K57R = self.DEFECT
                else:
                    K57R = self.COOP
            elif n == 28:
                K57R = self.COOP
            elif n < 32:
                K57R = self.DEFECT
            else:
                K57R = self.COOP

        elif n == 39:
            K57R = self.DEFECT

        else:
            if n < 45:
                K57R = self.COOP
            elif n == 45:
                K57R = self.DEFECT
            elif n < 49:
                K57R = self.COOP
            elif n == 49:
                K57R = self.DEFECT
            elif n < 58:
                K57R = self.COOP
            elif n == 58:
                K57R = self.DEFECT
            elif n < 61:
                K57R = self.COOP
            elif n == 61:
                K57R = self.DEFECT
            else:
                K57R = self.COOP

        self.last_move = K57R
        return K57R
    
class K58R(Strategy):
    def reset(self):
        self.KAM = 0
        self.NPHA = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        K = getattr(state, "score", 0)

        if M == 1:
            self.KAM = 0
            self.NPHA = 0

        if self.KAM > 6:
            return self.DEFECT

        if self.NPHA >= 1:
            self.NPHA -= 1
            if self.NPHA == 0:
                return self.DEFECT
            return self.COOP

        if (M % 18 == 0) and self.KAM > 2:
            self.KAM -= 1

        if M % 6 != 0:
            return self.COOP

        if K < M:
            self.KAM += 2
        if K * 10 < M * 15:
            self.KAM += 1
        if K < M * 2:
            self.KAM += 1
        if K * 10 < M * 25:
            self.KAM += 1

        self.NPHA = 2
        return self.DEFECT

class K60R(Strategy):
    def reset(self):
        self.ID = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        K = getattr(state, "score", 0)

        if M <= 1:
            self.ID = 0
            return self.COOP

        if self.ID != 1:
            K60R = J

            if M == 11 and K < 23:
                self.ID = 1
            elif M == 21 and K < 53:
                self.ID = 1
            elif M == 31 and K < 83:
                self.ID = 1
            elif M == 41 and K < 113:
                self.ID = 1
            elif M == 51 and K < 143:
                self.ID = 1
            elif M == 101 and K < 293:
                self.ID = 1

            return K60R

        return self.DEFECT

In [15]:
class K61R(Strategy):
    def reset(self):
        self.ICOOP = 0

    def act(self, state):
        ISPICK = state.history[-1][1] if state.history else 0
        ITURN = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if ITURN == 1:
            self.ICOOP = 0
            return self.COOP

        if ISPICK == 0:
            self.ICOOP += 1

        if ITURN <= 10:
            return self.COOP

        K61R = ISPICK

        if ITURN <= 25:
            return K61R

        K61R = self.COOP
        COPRAT = self.ICOOP / ITURN if ITURN > 0 else 0.0

        if ISPICK == 1 and COPRAT < 0.6 and R > COPRAT:
            K61R = self.DEFECT

        return K61R

class K62R(Strategy):
    def reset(self):
        self.JOLD = 0
        self.IRAN = 1

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if M == 1:
            self.JOLD = 0
            self.IRAN = int(23 * R) + 1

        K62R = self.COOP

        if M == self.IRAN:
            K62R = self.DEFECT
            self.IRAN = int(23 * R) + M + 1
        else:
            if self.JOLD == 1 and J == 1:
                K62R = self.DEFECT

        self.JOLD = J
        return K62R

class K63R(Strategy):
    def reset(self):
        self.ik = 1

    def act(self, state):
        M = len(state.history) + 1

        if M == 1:
            self.ik = 1

        self.ik = 1 - self.ik
        return self.ik

class K64R(Strategy):
    def reset(self):
        self.A = [[0, 0], [0, 0]]
        self.E = 0
        self.F = 0
        self.X = 1
        self.Y = 1

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.A = [[0, 0], [0, 0]]
            self.E = 0
            self.F = 0
            self.X = 1
            self.Y = 1
            K64R = self.COOP
            self.Y = K64R
            return K64R

        # Decide based on A(X,Y)
        if self.A[self.X - 1][self.Y - 1] >= 0:
            K64R = self.COOP
        else:
            K64R = self.DEFECT

        # Update matrix
        if J == 0:
            self.A[self.X - 1][self.Y - 1] += 1
        else:
            self.A[self.X - 1][self.Y - 1] -= 1

        # Update indices
        self.X = J + 1
        self.Y = K64R + 1

        # Track totals
        if J == 0:
            self.E += 1
        else:
            self.F += 1

        P = abs(self.E - self.F)

        if M > 40 and (10 * P < M):
            K64R = self.DEFECT

        return K64R

class K65R(Strategy):
    def reset(self):
        self.LASTD = 0
        self.DIFF = 0
        self.TOTD = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.LASTD = 0
            self.DIFF = 0
            self.TOTD = 0
            return self.COOP

        if self.TOTD >= 10:
            return self.DEFECT

        if J == 0:
            return self.COOP

        # opponent defect
        self.TOTD += 1

        if self.TOTD >= 10:
            return self.DEFECT

        if self.LASTD != 0:
            self.DIFF = M - self.LASTD
            if self.DIFF <= 4:
                self.TOTD = 10
                return self.DEFECT

        self.LASTD = M
        return self.COOP

In [16]:
class K66R(Strategy):
    def reset(self):
        self.D = 0
        self.J2 = -3

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.D = 0
            self.J2 = -3

        self.D += J

        RR = self.D / M if M > 0 else 0.0

        self.J2 = self.J2 - 1 + 3 * J
        self.J2 = max(-5, min(10, self.J2))

        if M < 3:
            return self.COOP

        if self.J2 < 3:
            return self.COOP

        if M <= 10:
            self.J2 = -1
            return self.DEFECT

        if RR < 0.15:
            return self.COOP

        return self.DEFECT
    
class K67R(Strategy):
    def reset(self):
        self.S = 0
        self.AD = 5
        self.NO = 0.0
        self.NK = 1.0
        self.AK = 1.0
        self.FD = 0
        self.C = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        K = getattr(state, "score", 0)
        L = getattr(state, "opponent_score", 0)
        R = state.random if hasattr(state, "random") else 0.0
        M = len(state.history) + 1

        if M == 1:
            self.reset()

        if self.FD == 2:
            self.FD = 0
            self.NO = (self.NO * self.NK + 3 - 3 * J + 2 * self.last_move - self.last_move * J) / (self.NK + 1)
            self.NK += 1

        if self.FD == 1:
            self.FD = 2
            self.AD = (self.AD * self.AK + 3 - 3 * J + 2 * self.last_move - self.last_move * J) / (self.AK + 1)
            self.AK += 1

        if J == 1:
            self.S += 1
        else:
            self.S = 0
            self.C += 1

        K67R = self.COOP

        # FD logic gate
        if abs(self.FD - 1.5) == 0.5:
            self.last_move = K67R
            return K67R

        if K >= 2.25 * M:
            P = 0.95 - (self.AD + self.NO - 5) / 15 + 1.0 / (M ** 2) - J / 4.0
            if R > P:
                K67R = self.DEFECT
                self.FD = 1
            self.last_move = K67R
            return K67R

        if K >= 1.75 * M:
            P = 0.25 + self.C / M - self.S * 0.25 + (K - L) / 100.0 + 4.0 / M
            if R > P:
                K67R = self.DEFECT
            self.last_move = K67R
            return K67R

        K67R = J

        self.last_move = K67R
        return K67R

class K68R(Strategy):
    def reset(self):
        self.J1 = 0
        self.J2 = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        R = state.random if hasattr(state, "random") else 0.0
        M = len(state.history) + 1

        if M == 1:
            self.J1 = 0
            self.J2 = 0
            return self.COOP

        if self.J1 * J == 1:
            K68R = self.COOP if R < 0.75 else self.DEFECT
            self.J2 = self.J1
            self.J1 = J
            return K68R

        if (self.J2 * 2 + self.J1 + J * 2 + J) == 1:
            K68R = self.DEFECT
            self.J2 = self.J1
            self.J1 = J
            return K68R

        if (self.J2 * 2 + self.J1 * 2 + J) == 1:
            if R < 0.5:
                K68R = self.COOP
            else:
                K68R = self.DEFECT

            self.J2 = self.J1
            self.J1 = J
            return K68R

        K68R = self.DEFECT

        self.J2 = self.J1
        self.J1 = J
        return K68R

class K69R(Strategy):
    def reset(self):
        self.S = 1
        self.F = 0
        self.D = 0
        self.C = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if M == 1:
            self.S = 1
            self.F = 0
            self.D = 0
            self.C = 0
            return self.COOP

        if J == 0:
            self.C += 1

        # state machine
        if self.S == 1:
            if R < 0.1:
                self.S = 5
                return self.DEFECT

            if J == 1:
                self.D += 1
            else:
                self.D = 0

            if self.D > 20:
                self.S = 3
                self.D = 0
                return self.COOP

            if self.C >= 0.7 * (M - 3):
                return J

            self.S = 2
            if J == 1:
                self.D += 1
            else:
                self.D = 0

            if self.D > 10:
                self.S = 3
                return self.COOP

            return self.DEFECT

        if self.S == 2:
            if J == 1:
                self.D += 1
            else:
                self.D = 0

            if self.D > 10:
                self.S = 3
                return self.DEFECT

            return self.DEFECT

        if self.S == 3:
            if J == 1:
                self.D += 1
            else:
                self.D = 0

            if self.D > 20:
                self.S = 3
                self.D = 0
                return self.COOP

            return J

        if self.S == 4:
            if J == 1:
                self.F += 1
                if self.F > 3:
                    self.S = 3
                    return self.COOP
            else:
                self.S = 1
                return self.COOP

        if self.S == 5:
            self.S = 4
            return self.act(state)

        return self.COOP

class K70R(Strategy):
    def reset(self):
        self.JZ = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        R = state.random if hasattr(state, "random") else 0.0
        M = len(state.history) + 1

        if M == 1:
            self.JZ = 0

        if self.JZ == J:
            K70R = self.JZ
        else:
            K70R = self.COOP
            if R > 0.2:
                K70R = self.DEFECT

        self.JZ = K70R
        return K70R

In [17]:
class K71R(Strategy):
    def reset(self):
        self.IA = 0
        self.IB = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.IA = 0
            self.IB = 0
            return self.COOP

        if M == 2:
            K71R = self.COOP
            if J == 1:
                K71R = self.DEFECT
            return K71R

        if J == 1:
            self.IB += 1
            if self.IB == 2:
                self.IB = 0
                return self.DEFECT
            return self.COOP

        self.IA += 1
        if self.IA == 2:
            self.IA = 0
            return self.DEFECT

        return self.COOP

class K72R(Strategy):
    def reset(self):
        self.JOLD = 0
        self.JCOUNT = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if M == 1:
            self.JOLD = 0
            self.JCOUNT = 0

        K72R = self.COOP

        self.JOLD = J
        if self.JOLD == 1:
            self.JCOUNT += 1

        N = 1.0
        if self.JOLD == 1 and M > 10:
            N = math.log(M)

        if R <= ((N * self.JCOUNT) / M):
            K72R = self.DEFECT

        return K72R



In [18]:
class K76R(Strategy):
    def reset(self):
        self.PATSY = True
        self.DC = 0
        self.MDC = 0
        self.G = 1

    def act(self, state):
        J = state.history[-1][1] if state.history else 0

        if len(state.history) + 1 == 1:
            self.PATSY = True
            self.DC = 0
            self.MDC = 0
            self.G = 1
            return self.DEFECT

        if not self.PATSY:
            return J

        if J != 1:
            self.DC += 1
            if self.G == 0:
                self.MDC += 1

            self.G = 0
            if self.MDC / (self.DC + 1) >= 0.5:
                self.G = 1

            return self.G

        # J == 1
        self.PATSY = False
        return self.COOP
    
class K77R(Strategy):
    def reset(self):
        self.KEXP = [100, 100, 100, 100, 100]
        self.JSTR = 3
        self.KTRY = 0
        self.KI = 0

    def act(self, state):
        JPICK = state.history[-1][1] if state.history else 0
        ISCORE = getattr(state, "score", 0)
        RANDOM = state.random if hasattr(state, "random") else 0.0
        MOVEN = len(state.history) + 1

        if MOVEN == 1:
            self.KEXP = [100, 100, 100, 100, 100]
            self.JSTR = 3
            self.KTRY = 0
            self.KI = 0

        # exploration / update every 20 tries
        if self.KTRY >= 20:
            self.KEXP[self.JSTR - 1] = ISCORE - self.KI

            if self.JSTR != 5:
                if self.KEXP[self.JSTR] <= self.KEXP[self.JSTR - 1]:
                    pass
                else:
                    self.JSTR += 1
                    JPICK = 0

            if self.JSTR == 5 or self.JSTR > 1:
                if self.KEXP[self.JSTR - 2] > self.KEXP[self.JSTR - 1]:
                    self.JSTR -= 1
                    JPICK = 0

            self.KI = ISCORE
            self.KTRY = 0

        self.KTRY += 1

        if self.JSTR == 1:
            return self.COOP

        if self.JSTR == 2:
            if JPICK == 0:
                return self.COOP
            return self.COOP if RANDOM <= 0.75 else self.DEFECT

        if self.JSTR == 3:
            return JPICK

        if self.JSTR == 4:
            if JPICK == 1:
                return self.DEFECT
            return self.COOP if RANDOM <= 0.75 else self.DEFECT

        return self.DEFECT

class K78R(Strategy):
    def reset(self):
        self.inner = GRASR()

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        K = getattr(state, "score", 0)
        L = getattr(state, "opponent_score", 0)
        R = state.random if hasattr(state, "random") else 0.0

        return self.inner.act(state)

class K79R(Strategy):
    def reset(self):
        self.JBACK = [0, 0, 0, 0, 0]

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.JBACK = [0, 0, 0, 0, 0]

        if M < 6:
            K79R = self.COOP
            self._update(J)
            return K79R

        I1 = sum(self.JBACK)

        if I1 >= 3:
            K79R = self.DEFECT
        else:
            K79R = self.COOP

        self._update(J)
        return K79R

    def _update(self, J):
        self.JBACK = self.JBACK[1:] + [J]

class K80R(Strategy):
    def reset(self):
        self.MODE = 0
        self.INOD = 0
        self.INOC = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.MODE = 0
            self.INOD = 0
            self.INOC = 0
            return self.COOP

        if self.MODE == 1:
            return self.DEFECT

        if J == 1:
            self.INOD += 1
        else:
            self.INOC = M - self.INOD

        INOC = M - self.INOD

        T1 = 1.6667 ** self.INOD
        T2 = 0.882 ** INOC
        TEST = T1 * T2

        if TEST >= 5.0:
            self.MODE = 1
            return self.DEFECT

        return self.COOP

In [19]:
class K81R(Strategy):
    def reset(self):
        self.L4 = [[0.0 for _ in range(2)] for _ in range(8)]
        self.X = [0] * 8

        self.T0 = 0
        self.T4 = 0
        self.T5 = 0
        self.T6 = 25
        self.T8 = 0
        self.T9 = 5

        self.D4 = 0
        self.A = 0.0
        self.B = 0.0
        self.S1 = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        K = getattr(state, "score", 0)
        L = getattr(state, "opponent_score", 0)
        M = len(state.history) + 1

        if M == 81 and K == L == 237:
            self.T0 = 1

        if M == 1:
            self.L4 = [[0.0, 0.0] for _ in range(8)]
            self.T0 = 0
            self.T4 = 0
            self.T5 = 0
            self.T6 = 25
            self.T8 = 0
            self.T9 = 5
            self.D4 = 0
            self.A = 0.0
            self.B = 0.0
            self.S1 = 0
            self.X = [0] * 8

        if M == 2 and J == 1:
            self.T9 = 9

        if M < self.T9:
            return self.COOP

        if self.T5 > 7:
            self.T5 -= 8

        if J == 0:
            self.L4[self.T5 % 8][0] += 1

        if self.T9 == 9 and self.T0 == 1:
            pass  # special branch in original (skipped detailed subroutine)

        # --- main decision path simplified from FORTRAN control flow ---
        if L <= K + self.T6:
            self.D4 = self.T4 % 8

            A1 = self.L4[self.D4][0]
            A2 = self.L4[self.D4][1] if self.L4[self.D4][1] != 0 else 1.0

            A3 = A1 / A2
            self.A = 3 * A3
            self.B = self.A + A3 + 1

            # heuristic selection (X-array logic collapsed)
            X_scores = [self.A] * 4 + [self.B] * 4
            self.S1 = max(range(8), key=lambda i: X_scores[i])

            # simplified mapping of S1 outcome
            K81R = self.DEFECT if self.S1 < 5 else self.COOP
        else:
            K81R = J

        # update state
        self.T5 = self.T4

        if M % 10 == 0:
            for c in range(8):
                self.L4[c][0] *= 9
            self.T6 += 1

        # update T4 encoding
        if self.T4 > 4:
            self.T4 -= 4

        self.T4 = self.T4 * 2 + K81R

        return K81R
    
class K82R(Strategy):
    def reset(self):
        self.X = 0.75
        self.I5 = 0
        self.D4 = 0.0
        self.I1 = 0
        self.I2 = 0
        self.I3 = 1

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = state.random if hasattr(state, "random") else 0.0

        if M == 1:
            self.X = 0.75
            self.I5 = 0
            self.D4 = 0.0
            self.I1 = 0
            self.I2 = 0
            self.I3 = 1

        # initial update (always overwrites in FORTRAN)
        K82R = J

        if J == 0 and self.I5 > 1:
            self.I5 = 0
            # jumps to 2010 in FORTRAN

        if M < 30:
            return K82R

        if self.I3 == 0:
            return self.COOP

        if abs(self.D4 / (M - 1.0) - 0.5) < 0.1:
            self.X -= 0.2

        if self.I2 == 1:
            if J == 1:
                self.X = min(1.0, self.X + 0.15)
            return self.COOP

        if R > self.X:
            self.I1 = 1
            return self.DEFECT

        if J == 1:
            self.X = max(0.0, self.X - 0.05)
            self.I2 = 0
            if self.X < 0.3:
                return self.DEFECT
            return K82R

        # default transitions
        self.I2 = 0
        self.I3 = 1 if self.I5 <= 5 else 0

        self.I5 = 0
        self.I1 = 0

        return self.COOP
    
class K83R(Strategy):
    def reset(self):
        self.JHIS = [0, 0, 0, 0, 0]
        self.JTOT = 0
        self.MCNT = 0

    def act(self, state):
        JPICK = state.history[-1][1] if state.history else 0
        MOVEN = len(state.history) + 1
        RAND = state.random if hasattr(state, "random") else 0.0

        if MOVEN == 1:
            self.JHIS = [0, 0, 0, 0, 0]
            self.JTOT = 0
            self.MCNT = 1

        if MOVEN <= 5:
            if MOVEN == 1:
                self.JTOT = 0
                self.MCNT = 1

            self.JHIS[MOVEN - 1] = JPICK
            self.JTOT += JPICK
            return self.COOP

        # sliding window update
        self.JTOT = self.JTOT - self.JHIS[self.MCNT - 1] + JPICK
        self.JHIS[self.MCNT - 1] = JPICK

        self.MCNT += 1
        if self.MCNT > 5:
            self.MCNT = 1

        K83R = self.COOP

        if RAND * 25 < (self.JTOT * self.JTOT - 1):
            K83R = self.DEFECT

        return K83R

import math

class K84R(Strategy):
    def reset(self):
        self.ISIG = 0
        self.DS = 0
        self.JQ = 0
        self.FJD = 0
        self.JDR = 0
        self.FM = 0

    def act(self, state):
        JP = state.history[-1][1] if state.history else 0
        IS = getattr(state, "score", 0)
        JS = getattr(state, "opponent_score", 0)
        M = len(state.history) + 1

        K84R = self.DEFECT

        if M == 1:
            self.ISIG = 0
            self.DS = 0
            self.JQ = 0
            self.FJD = 0
            self.JDR = 0
            self.FM = 0
            return self.DEFECT

        if JP == 1:
            self.FJD += 1

        if self.ISIG == 0:
            self.FM = M

            if self.JQ == 0 and JP == 1:
                self.JDR += 1

            if IS - JS - self.DS - 5 * self.JDR * (self.JDR - 1) / 2 >= 0:
                K84R = self.COOP
            else:
                self.JQ = JP
                return self.DEFECT

            if (self.JQ - JP) < 0 or (M - self.ISIG) < 10:
                self.JQ = JP
                return K84R

            if abs(self.FJD - (self.FM - 1) / 2.0) < (1.5 * math.sqrt(self.FM - 1)):
                self.ISIG = 1
                self.JQ = JP
                return self.DEFECT

            self.JQ = JP
            return K84R

        # ISIG != 0
        self.ISIG = M
        self.JQ = 0
        self.JDR = 0
        self.DS = IS - JS
        return self.DEFECT

class K85R(Strategy):
    def reset(self):
        self.J2 = 0
        self.J4 = 0
        self.J8 = 0
        self.J0 = 0
        self.F4 = 0
        self.F8 = 0
        self.F0 = 0
        self.F1 = 0
        self.C = 0
        self.D = 0
        self.T = 0

        self.I1 = 0
        self.I2 = 0
        self.I3 = 0
        self.I4 = 0

    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if M == 1:
            self.reset()
            return self.COOP

        # --- shift register update ---
        J5 = self.J0 / 1e7
        J3 = int(J5)
        self.J8 = (J5 - J3) * 1e7

        F5 = self.F0 / 1e7
        F3 = int(F5)
        self.F8 = (F5 - F3) * 1e7

        self.J0 = self.J8 * 10 + 5
        self.F0 = self.F8 * 10 + 5

        # --- counters ---
        if self.F1 == 0:
            if J == 0:
                self.I3 += 1
            else:
                self.I4 += 1
        else:
            if J == 0:
                self.I1 += 1
            else:
                self.I2 += 1

        # --- randomness check ---
        if M > 20:
            A = (self.I1 + 1e-6) / (self.I2 + 1e-6)
            B = (self.I3 + 1e-6) / (self.I4 + 1e-6)

            if not (0.5 <= A <= 1.5 and 0.5 <= B <= 1.5):
                return self.DEFECT

        # --- mode checks ---
        if self.T == 1:
            return J

        if self.J0 == 11111111:
            return self.DEFECT

        if self.C == 1:
            return self._handle_C_mode(J)

        # --- tit-for-tat detection ---
        if M <= 30:
            J2 = (self.J0 % 100) // 1
            if J2 == 11:
                return self._enter_CC_first()

        J4 = (self.J0 // 10000) % 10000
        F4 = (self.F0 // 10000) % 10000

        if J4 == 1011 and F4 == 111:
            return self._enter_tft()

        # defect balance check
        Y1 = self.I1 + self.I2
        Y2 = self.I1 + self.I2 + 3
        if Y1 >= Y2:
            return self.DEFECT

        # --- default rules (collapsed) ---
        if self.D == 1:
            return self._return_dc(J)

        if self.F1 == 0:
            if J == 0:
                return self.COOP
        else:
            if J == 1:
                return self.DEFECT

        return self.COOP

    # --- helper placeholders (original had multiple GOTOs) ---
    def _enter_tft(self):
        self.T = 1
        return self.COOP

    def _enter_CC_first(self):
        self.C = 1
        return self.DEFECT

    def _handle_C_mode(self, J):
        self.C = 0
        self.F1 = 0
        return self.COOP

    def _return_dc(self, J):
        self.F1 = 0
        self.D = 0
        return self.DEFECT

In [20]:
class K86R(Strategy):
    def act(self, state):
        if not hasattr(self, "IOPPNT"):
            self.IOPPNT = {}

        J = state.history[-1][1] if state.history else 0
        MOVEN = len(state.history) + 1
        RANDOM = random.random()

        self.IOPPNT[MOVEN] = J

        MYOLD = getattr(self, "last", 0)

        if MOVEN <= 2:
            self.last = 0
            return 0

        if MOVEN <= 7:
            self.last = J
            return J

        IPREV7 = 0
        for i in range(MOVEN - 7, MOVEN):
            IPREV7 += self.IOPPNT.get(i, 0)

        if MYOLD == 0 and IPREV7 <= 2:
            self.last = 0
        elif MYOLD == 0 and IPREV7 > 2:
            self.last = 1
        elif MYOLD == 1 and IPREV7 <= 1:
            self.last = 0
        elif MYOLD == 1 and IPREV7 > 1:
            self.last = 1

        return self.last

class K87R(Strategy):
    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = random.random()

        if not hasattr(self, "init"):
            self.init = True
            self.Z = 0
            self.Q6 = 0.5
            self.S = 0
            self.H = 0

        if M == 1:
            self.Z = 0
            self.Q6 = 0.5
            self.S = 0
            self.H = 0
            return 0

        self.S = 2 * J + self.H + 1

        if self.Z == 0:
            if J == 1:
                self.Z = 1

        if self.S <= 1:
            self.Q6 = self.Q6 * 0.57 + 0.43
        elif self.S == 4:
            self.Q6 = 0.74 * self.Q6 + 0.104
        else:
            self.Q6 = 0.5 * self.Q6

        self.H = 1

        if R > self.Q6:
            return 1

        self.H = 0
        return 0

class K88R(Strategy):
    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1
        R = random.random()

        if not hasattr(self, "init"):
            self.init = True
            self.MMC = 0
            self.LMV = 0
            self.MP = 0
            self.MMV = 0
            self.MP2 = 0
            self.MMD = 1
            self.DFLG = 0
            self.PRC = 0.0
            self.PRD = 0.0

        self.init = True

        K88R = 0

        if M == 1:
            self.MMC = 0
            self.LMV = 0
            self.MP = 0
            self.MMV = 0
            self.MP2 = 0
            self.MMD = 1
            self.DFLG = 0

        if M >= 2:
            if self.MMV != 0:
                self.MMD += 1
                self.MP2 += J
                self.PRD = float(self.MP2) / float(self.MMD)
            else:
                self.MMC += 1
                self.MP += J
                self.PRC = float(self.MP) / float(self.MMC)

        if M > 4:
            if J == 1 and self.DFLG == 0:
                self.DFLG = 1
                K88R = 0
            else:
                if self.MMV == 0 and R < self.PRC:
                    K88R = 1
                if self.MMV == 1 and R < self.PRD:
                    K88R = 1

        self.MMV = self.LMV
        self.LMV = K88R

        return K88R

class K89R(Strategy):
    def act(self, state):
        HCM = state.history[-1][1] if state.history else 0
        MN = len(state.history) + 1
        MYSC = getattr(self, "MYSC", 0)

        if not hasattr(self, "init"):
            self.init = True
            self.SC = [0] * 6
            self.SL = [1] * 6
            self.ST = [0] * 5
            self.GT = [0] * 5
            self.TM = [0] * 6
            self.CN = 10
            self.TM[5] = 0
            self.SL[5] = 1
            self.CSRC = 5
            self.MYLM = 1
            self.HLM = 0

        while True:
            CODE = self.CN // 10

            # FIX: prevent out-of-range indexing
            if CODE < 0 or CODE >= len(self.SL):
                self.CN += 10
                continue

            if 10 * CODE == self.CN:
                self.SC[CODE] = MYSC

            if self.SL[CODE] == 1:
                self.CN += 1
                self.TM[CODE] += 1

                if CODE == 1:
                    return 0
                elif CODE == 2:
                    return 1
                elif CODE == 3:
                    self.MYLM = 1 - self.MYLM
                    return self.MYLM
                elif CODE == 4:
                    return 1 if HCM == 1 else 0
                elif CODE == 5:
                    if HCM == 1 and self.HLM == 1:
                        return 1
                    self.HLM = HCM
                    return 0
                elif CODE == 6:
                    SGT = 0
                    for i in range(5):
                        self.ST[i] = self.SC[i + 1] - self.SC[i]
                        SGT += self.ST[i]
                        self.GT[i] += self.ST[i]

                    MEAN = SGT / self.CSRC if self.CSRC != 0 else 0
                    AMEAN = 9 * MEAN / 10

                    self.CSRC = 0

                    for i in range(5):
                        if self.SL[i] == 1:
                            if self.ST[i] < AMEAN:
                                self.SL[i] = 0
                            else:
                                if self.TM[i] != 0 and (10 * self.GT[i] / self.TM[i]) > AMEAN:
                                    self.SL[i] = 1

                        if self.SL[i] == 1:
                            self.CSRC += 1

                    self.CN = 10
                    continue

            self.CN += 10

class K90R(Strategy):
    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if not hasattr(self, "init"):
            self.init = True
            self.jold = 0

        if M == 1:
            self.jold = 0

        K90R = 0

        if self.jold == 1 and J == 1:
            K90R = 1

        self.jold = J

        return K90R

In [21]:
class K91R(Strategy):
    def act(self, state):
        J = state.history[-1][1] if state.history else 0
        M = len(state.history) + 1

        if not hasattr(self, "init"):
            self.init = True

            self.X = 0.999
            self.PX = 0.001
            self.Y = 0.001
            self.PY = 0.999
            self.Z = 0.999
            self.PZ = 0.001
            self.W = 0.001
            self.PW = 0.999

            self.QC = [1.999, 1.999, 0.001, 0.001]
            self.QN = [2, 2, 2, 2]

            self.E = [0] * 11

            self.IPOL = [
                [0, 1, 1, 0],
                [1, 1, 1, 0],
                [1, 1, 0, 0],
                [1, 0, 0, 1],
                [1, 1, 1, 0],
                [0, 1, 1, 0],
                [1, 0, 0, 1],
                [0, 1, 0, 0],
                [1, 1, 0, 1],
                [1, 0, 0, 1],
                [0, 0, 1, 1],
            ]

            self.IOLD = 0
            self.N = 0

            return 0

        # update stats
        if M > 2:
            if self.N >= 0:
                if J == 0:
                    self.QC[self.N] += 1
                self.QN[self.N] += 1

        if self.N == 0:
            self.X = self.QC[0] / self.QN[0]
            self.PX = 1 - self.X
        elif self.N == 1:
            self.Z = self.QC[1] / self.QN[1]
            self.PZ = 1 - self.Z
        elif self.N == 2:
            self.Y = self.QC[2] / self.QN[2]
            self.PY = 1 - self.Y
        elif self.N == 3:
            self.W = self.QC[3] / self.QN[3]

        X, PX = self.X, self.PX
        Y, PY = self.Y, self.PY
        Z, PZ = self.Z, self.PZ
        W = self.W
        PW = self.PW

        E = [0] * 11

        E[0] = (3 * Z) / (Z + PX + 1e-9)
        E[1] = (3 * (Y * Z + W * PZ) + 5 * Z * PX + PX * PZ) / (Y * Z + W * PZ + PX + Z * PX + PX * PZ + 1e-9)
        E[2] = (3 * W * Y + 5 * W * PX + PX * PZ) / (W * Y + 2 * W * PX + PX * PZ + 1e-9)
        E[3] = (3 * W * PY + 5 * Z * PX + PX * PY) / (W * PY + PX * PY + Z * PX + PX * PY + 1e-9)
        E[4] = (3 * Z + 5 * X * Z + Z * PX) / (1 - X * Y - W * PX + 2 * Z + 1e-9)
        E[5] = (8 * W * Z + Z * PX) / (2 * W * Z + W * PY + Z * PX + 1e-9)
        E[6] = (3 * Z * PY + 5 * X * Z + Z * PY) / (2 * Z * PY + PW * PY + X * Z + 1e-9)
        E[7] = (3 * (Y * Z + W * PZ) + 5 * (Z * PW + W * X) + 1 - X * Y - Z * PY) / (Y * Z + W * PZ + 2 - 2 * X * Y - W * PX + Z * PW + W * X - Z * PY + 1e-9)
        E[8] = (3 * W * Y + 5 * W + 1 - X * Y - Z * PY) / (2 * W + 1 - X * Y - Z * PY + 1e-9)
        E[9] = (3 * W * PY + 5 * (Z * PW + W * X) + PY) / (PY + Z * PW + W * X + PY + 1e-9)
        E[10] = (5 * W + PY) / (W + PY + 1e-9)

        ibest = max(range(11), key=lambda i: E[i])

        IOLD = getattr(self, "last", 0)
        N = 2 * IOLD + J

        action = self.IPOL[ibest][N]

        self.N = N
        self.last = action
        return action
    
class K93R(Strategy):
    def act(self, state):
        import random

        return self.COOP if random.random() >= 0.5 else self.DEFECT

## Game Match Simulations

In [22]:
import os
import json
from concurrent.futures import ProcessPoolExecutor

# make sure output folder exists
os.makedirs("logs", exist_ok=True)


# ----------------------------
# WORKER FUNCTION (parallel)
# ----------------------------
def run_match(pair):
    S1, S2 = pair

    # fresh instances (VERY IMPORTANT for correctness)
    s1 = S1()
    s2 = S2()

    state = MatchState(N=5, T=100)

    games = state.match_play(s1, s2)

    name1 = S1.__name__
    name2 = S2.__name__

    # save full trajectory per match
    filepath = f"logs/{name1}_vs_{name2}.json"

    with open(filepath, "w") as f:
        json.dump({
            "player1": name1,
            "player2": name2,
            "games": games
        }, f)

    print(f"Saved: {name1} vs {name2}")

    # return summary only (lightweight)
    return {
        "player1": name1,
        "player2": name2,
        "n_games": len(games)
    }


# ----------------------------
# MAIN EXECUTION
# ----------------------------
def run_tournament(strategy_classes, N=5, T=100, workers=None):

    pairs = [(S1, S2) for S1 in strategy_classes for S2 in strategy_classes]

    results = []

    with ProcessPoolExecutor(max_workers=workers) as executor:
        for result in executor.map(run_match, pairs):
            results.append(result)
            print(f"{result['player1']} vs {result['player2']}: {result['n_games']} games")

    # ----------------------------
    # FIX: JSON cannot use tuple keys
    # store as LIST instead (correct format)
    # ----------------------------
    summary_path = "logs/summary.json"

    with open(summary_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\nTournament complete. Summary saved to {summary_path}")

    return results


# ----------------------------
# USAGE
# ----------------------------
if __name__ == "__main__":

    strategy_classes = [
        TitForTat, TitForTwoTats, GrimTrigger, Pavlov,
        K59R, K73R, K74R, K74RXX, K75R, K92R,
        KPavlovC, KRandomC, KTF2TC, KTitForTatC, GRASR,
        K31R, K32R, K33R, K34R, K35R, K36R, K37R, K38R, K39R,
        K40R, K41R, K42R, K43R, K44R, K45R, K46R, K47R, K48R,
        K49R, K50R, K51R, K52R, K53R, K54R, K55R, K56R, K57R,
        K58R, K60R, K61R, K62R, K63R, K64R, K65R, K66R, K67R,
        K68R, K69R, K70R, K71R, K72R, K76R, K77R, K78R, K79R,
        K80R, K81R, K82R, K83R, K84R, K85R, K86R, K87R, K88R,
        K90R, K91R, K92R, K93R
    ]

    run_tournament(strategy_classes, N=5, T=100, workers=None)

Saved: TitForTat vs TitForTat
Saved: TitForTat vs TitForTwoTatsSaved: TitForTat vs PavlovSaved: TitForTat vs GrimTrigger


Saved: TitForTat vs K73RSaved: TitForTat vs K74RXXSaved: TitForTat vs K59RSaved: TitForTat vs KPavlovCSaved: TitForTat vs K92R




Saved: TitForTat vs K74R
Saved: TitForTat vs K75RSaved: TitForTat vs KRandomCSaved: TitForTat vs KTitForTatCSaved: TitForTat vs KTF2TC



Saved: TitForTat vs K35RSaved: TitForTat vs K34RSaved: TitForTat vs K32R

Saved: TitForTat vs GRASRSaved: TitForTat vs K37R
Saved: TitForTat vs K33R



Saved: TitForTat vs K36RSaved: TitForTat vs K31RSaved: TitForTat vs K38RSaved: TitForTat vs K40R
Saved: TitForTat vs K42R
Saved: TitForTat vs K41R


Saved: TitForTat vs K43RSaved: TitForTat vs K45R
Saved: TitForTat vs K44RSaved: TitForTat vs K39R


Saved: TitForTat vs K46R
Saved: TitForTat vs K50RSaved: TitForTat vs K51RSaved: TitForTat vs K52RSaved: TitForTat vs K54RSaved: TitForTat vs K47R
Saved: TitForTat vs K49R



Saved: TitForTat vs K53RSaved: Ti

In [24]:
import json
import pandas as pd
import glob


def summarize_game(history):
    # history: [[a1,a2,r1,r2], ...]

    T = len(history)

    a1 = [h[0] for h in history]
    a2 = [h[1] for h in history]
    r1 = [h[2] for h in history]
    r2 = [h[3] for h in history]

    # cooperation rate
    p1_coop = sum(a1) / T
    p2_coop = sum(a2) / T

    # average reward
    p1_avg_r = sum(r1) / T
    p2_avg_r = sum(r2) / T

    # phase changes (behavior instability)
    changes = 0
    for i in range(1, T):
        if a1[i] != a1[i-1] or a2[i] != a2[i-1]:
            changes += 1

    return {
        "T": T,
        "p1_coop_rate": p1_coop,
        "p2_coop_rate": p2_coop,
        "p1_avg_reward": p1_avg_r,
        "p2_avg_reward": p2_avg_r,
        "phase_changes": changes
    }


def load_game_level_df(log_dir="logs"):
    rows = []

    for file in glob.glob(f"{log_dir}/*.json"):
        if "summary" in file:
            continue

        with open(file, "r") as f:
            data = json.load(f)

        p1 = data["player1"]
        p2 = data["player2"]

        for game_id, game in enumerate(data["games"]):
            stats = summarize_game(game["history"])

            rows.append({
                "match_id": f"{p1}_vs_{p2}",
                "game_id": game_id,
                "player1": p1,
                "player2": p2,
                **stats
            })

    return pd.DataFrame(rows)


df = load_game_level_df("logs")
print(df.head())

       match_id  game_id player1 player2    T  p1_coop_rate  p2_coop_rate  \
0  K31R_vs_K46R        0    K31R    K46R  100          0.87          0.69   
1  K31R_vs_K46R        1    K31R    K46R  100          0.26          0.14   
2  K31R_vs_K46R        2    K31R    K46R  100          0.03          0.09   
3  K31R_vs_K46R        3    K31R    K46R  100          0.02          0.00   
4  K31R_vs_K46R        4    K31R    K46R  100          0.04          0.33   

   p1_avg_reward  p2_avg_reward  phase_changes  
0           1.49           0.95              6  
1           1.98           1.62              4  
2           1.85           2.03              6  
3           2.02           1.96              3  
4           1.38           2.25              5  
